In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| urządzenie:", device)


mean, std = 0.2860, 0.3530

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])
train_tf_plain = test_tf
train_tf_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])

train_plain = torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=train_tf_plain)
train_aug   = torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=train_tf_aug)
test_ds     = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=test_tf)
classes = test_ds.classes

def loaders(train_ds, bs=128):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True),
            DataLoader(test_ds,  batch_size=256, shuffle=False))


class MLP(nn.Module):
    def __init__(self, in_dim=28*28, hidden=(256, 128), n_classes=10,
                 use_bn=False, p_drop=0.0):
        super().__init__()
        layers, d = [nn.Flatten()], in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if p_drop > 0:
                layers.append(nn.Dropout(p_drop))
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


def evaluate(model, dl):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    crit = nn.CrossEntropyLoss(reduction="sum")
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss_sum += crit(out, yb).item()
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.numel()
    return loss_sum/total, correct/total

def run_training(model, train_dl, test_dl, epochs=10, lr=1e-3,
                 weight_decay=0.0, optimizer="adam", use_scheduler=False, verbose=True):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    if optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs//3), gamma=0.5) if use_scheduler else None
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}
    n = len(train_dl.dataset)
    for epoch in range(1, epochs+1):
        model.train()
        run = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item()*yb.numel()
        if sched: sched.step()
        tr = run/n
        te, acc = evaluate(model, test_dl)
        hist["train_loss"].append(tr); hist["test_loss"].append(te); hist["test_acc"].append(acc)
        if verbose:
            print(f"Epoka {epoch:2d} | trening {tr:.3f} | test {te:.3f} | accuracy {acc*100:.2f}%")
    return hist


EPOCHS = 12

print("== BASELINE ==")
tr_dl, te_dl = loaders(train_plain)
base = MLP(use_bn=False, p_drop=0.0)
hist_base = run_training(base, tr_dl, te_dl, epochs=EPOCHS, lr=1e-3, weight_decay=0.0, use_scheduler=False)

print("\n== STROJONY ==")
tr_dl_a, te_dl = loaders(train_aug)
tuned = MLP(use_bn=True, p_drop=0.3)
hist_tuned = run_training(tuned, tr_dl_a, te_dl, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4, use_scheduler=True)

print(f"\nNajlepsze accuracy — baseline: {max(hist_base['test_acc'])*100:.2f}% | strojony: {max(hist_tuned['test_acc'])*100:.2f}%")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h, c in [("baseline", hist_base, "tab:red"), ("strojony", hist_tuned, "tab:green")]:
    ax1.plot(h["train_loss"], "--", color=c, label=f"{name} — trening")
    ax1.plot(h["test_loss"],  "-",  color=c, label=f"{name} — test")
    ax2.plot([a*100 for a in h["test_acc"]], color=c, label=name)
ax1.set_xlabel("epoka"); ax1.set_ylabel("strata"); ax1.legend(fontsize=8); ax1.set_title("Krzywe straty")
ax2.set_xlabel("epoka"); ax2.set_ylabel("accuracy [%]"); ax2.legend(); ax2.set_title("Accuracy (test)")
plt.tight_layout(); plt.show()

for name, h in [("baseline", hist_base), ("strojony", hist_tuned)]:
    gap = h["test_loss"][-1] - h["train_loss"][-1]
    print(f"{name:9s} | przerwa test-trening (strata): {gap:.3f} | accuracy: {h['test_acc'][-1]*100:.2f}%")


lrs = [1e-1, 1e-2, 1e-3, 1e-4]
acc_by_lr = []
tr_dl, te_dl = loaders(train_plain)
for lr in lrs:
    m = MLP(use_bn=False, p_drop=0.0)
    h = run_training(m, tr_dl, te_dl, epochs=4, lr=lr, use_scheduler=False, verbose=False)
    acc_by_lr.append(max(h["test_acc"]))
    print(f"lr={lr:<7} -> accuracy {max(h['test_acc'])*100:.2f}%")

plt.figure(figsize=(6,4))
plt.semilogx(lrs, [a*100 for a in acc_by_lr], "o-")
plt.xlabel("learning rate (log)"); plt.ylabel("accuracy [%]"); plt.title("Wpływ LR (4 epoki)")
plt.grid(True, which="both", alpha=0.3); plt.show()

PyTorch: 2.11.0+cpu | urządzenie: cpu


100%|██████████| 26.4M/26.4M [00:06<00:00, 4.34MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 303kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.9MB/s]


== BASELINE ==
Epoka  1 | trening 0.489 | test 0.414 | accuracy 84.79%
Epoka  2 | trening 0.351 | test 0.355 | accuracy 87.17%
Epoka  3 | trening 0.312 | test 0.356 | accuracy 87.25%
Epoka  4 | trening 0.286 | test 0.345 | accuracy 87.54%
Epoka  5 | trening 0.267 | test 0.343 | accuracy 87.52%
Epoka  6 | trening 0.253 | test 0.333 | accuracy 88.18%
Epoka  7 | trening 0.234 | test 0.320 | accuracy 88.60%
Epoka  8 | trening 0.221 | test 0.334 | accuracy 88.62%
Epoka  9 | trening 0.211 | test 0.334 | accuracy 88.28%
Epoka 10 | trening 0.200 | test 0.379 | accuracy 88.16%
